In [25]:
import pandas as pd
import numpy as np

# 0. 데이터 로드
df = pd.read_csv('marketing_campaign.csv', sep=';')
print(f"[0] 원본 로드: {df.shape}")

[0] 원본 로드: (2240, 29)


In [26]:
#  ============================================================
# 1. 중복 제거 (완전중복 먼저 → Response만 다른 쌍 나중)
#    전체가 다 동일한 경우 → 한 개만 남기기 (연루 358 / 제거대상 182)
#    ID/Response만 다르고 나머지 전부 동일한 경우 → Response=1로 남기기
# ============================================================
 
# 1-1. ID 제외 전부(Response 포함) 동일한 완전중복 → 한 개만 남기기
cols_except_id = [c for c in df.columns if c != 'ID']
before = len(df)
df = df.drop_duplicates(subset=cols_except_id, keep='first').reset_index(drop=True)
print(f"[1-1] 완전중복 제거 후: {df.shape} (제거된 행: {before - len(df)})")
 
# 1-2. ID/Response만 다르고 나머지 27개 컬럼이 전부 동일한 그룹 처리
#      → 이 그룹에서는 Response=1인 행을 우선 채택
cols_except_id_response = [c for c in df.columns if c not in ['ID', 'Response']]
before = len(df)
df = (
    df.sort_values('Response', ascending=False)  # Response=1이 먼저 오도록
      .drop_duplicates(subset=cols_except_id_response, keep='first')
      .reset_index(drop=True)
)
print(f"[1-2] ID/Response만 다른 중복 처리 후: {df.shape} (제거된 행: {before - len(df)})")

[1-1] 완전중복 제거 후: (2058, 29) (제거된 행: 182)
[1-2] ID/Response만 다른 중복 처리 후: (2039, 29) (제거된 행: 19)


In [27]:
# ============================================================
# 2. 불필요한 컬럼 제거 (Z_CostContact, Z_Revenue — 전 행 상수값)
# ============================================================
df = df.drop(columns=['Z_CostContact', 'Z_Revenue'])
print(f"[2] Z_ 컬럼 제거 후: {df.shape}")

[2] Z_ 컬럼 제거 후: (2039, 27)


In [28]:
# ============================================================
# 3. Year_Birth → 1940년대 이후만 남기기 (1893/1899/1900 등 이상치 제거)
#    나이 계산은 2014년 기준
# ============================================================
before = len(df)
df = df[df['Year_Birth'] >= 1940].reset_index(drop=True)
print(f"[3] Year_Birth 이상치 제거 후: {df.shape} (제거된 행: {before - len(df)})")
 
REFERENCE_YEAR = 2014
df['Age'] = REFERENCE_YEAR - df['Year_Birth']

[3] Year_Birth 이상치 제거 후: (2036, 27) (제거된 행: 3)


In [29]:
# 나이 범주형 파생 (나중에 세그먼트 설명용, 분석은 연속형 Age 사용 권장)
age_bins = [0, 30, 40, 50, 60, 70, 150]
age_labels = ['20대이하', '30대', '40대', '50대', '60대', '70대이상']
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

In [30]:
print(df.shape)          # 이제는 28개 나와야 함 (27 + Age)
print('Age' in df.columns)

(2036, 29)
True


In [31]:
# ============================================================
# 4. Dt_Customer → datetime 변환
# ============================================================
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%Y-%m-%d')
print(f"[4] Dt_Customer datetime 변환 완료 (dtype: {df['Dt_Customer'].dtype})")

[4] Dt_Customer datetime 변환 완료 (dtype: datetime64[us])


In [32]:
# ============================================================
# 5. Education → 4개 항목으로 정리 (2n Cycle → Master로 통합)
# ============================================================
df['Education'] = df['Education'].replace({'2n Cycle': 'Master'})
print(f"[5] Education 정리 후 항목:\n{df['Education'].value_counts()}")

[5] Education 정리 후 항목:
Education
Graduation    1025
Master         521
PhD            441
Basic           49
Name: count, dtype: int64


In [33]:
# ============================================================
# 6. Marital_Status → Alone/Absurd/YOLO는 Single로,
#    Divorced/Widow는 그대로 유지
# ============================================================
df['Marital_Status'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
})
print(f"[6] Marital_Status 정리 후 항목:\n{df['Marital_Status'].value_counts()}")

[6] Marital_Status 정리 후 항목:
Marital_Status
Married     788
Together    516
Single      450
Divorced    212
Widow        70
Name: count, dtype: int64


In [34]:
# ============================================================
# 7. Income 결측 + 이상치(666666 등) → 중앙값 대체
#    * Income_isnull: 결측 여부 플래그 컬럼 별도 생성 (분석 시 결측이었음을 추적 가능하게)
# ============================================================
df['Income_isnull'] = df['Income'].isna().astype(int)

In [35]:
# 7. Income 결측 + 이상치(666666 단일값만 지정) → 중앙값 대체
#    * Income_isnull: 결측 여부 플래그 컬럼 별도 생성 (분석 시 결측이었음을 추적 가능하게)
# ============================================================
df['Income_isnull'] = df['Income'].isna().astype(int)
 
# 이상치 정의: 666666 단일값만 이상치로 지정 (IQR 등 통계적 기준 미적용)
OUTLIER_VALUE = 666666
n_outlier = (df['Income'] == OUTLIER_VALUE).sum()
n_missing = df['Income'].isna().sum()
print(f"[7] Income 결측치: {n_missing}건, 이상치(={OUTLIER_VALUE}): {n_outlier}건 → 둘 다 중앙값으로 대체")
 
# 중앙값은 결측/이상치를 제외한 정상값 기준으로 계산
income_median = df.loc[df['Income'] != OUTLIER_VALUE, 'Income'].median()
df.loc[df['Income'] == OUTLIER_VALUE, 'Income'] = np.nan  # 이상치를 결측으로 표시 후 대체
df['Income'] = df['Income'].fillna(income_median)
print(f"[7] Income 중앙값({income_median}) 대체 완료, 결측 남은 개수: {df['Income'].isna().sum()}")

# ※ 참고: Income=7500이 12명 반복되는 현상 확인함.
#   원 제작자의 결측치 임의 대체(imputation) 가능성이 있으나,
#   나머지 특성(생년/학력/자녀수 등)이 전부 다른 별개의 인물로 확인되어
#   실제 소득일 가능성을 배제할 수 없음.
#   → 별도 처리 없이 그대로 유지, 주석으로만 기록.

[7] Income 결측치: 24건, 이상치(=666666): 1건 → 둘 다 중앙값으로 대체
[7] Income 중앙값(51529.0) 대체 완료, 결측 남은 개수: 0


In [36]:
# ============================================================
# 8. Kidhome + Teenhome → Children 파생변수
# ============================================================
df['Children'] = df['Kidhome'] + df['Teenhome']
print(f"[8] Children 분포:\n{df['Children'].value_counts().sort_index()}")

[8] Children 분포:
Children
0     573
1    1034
2     381
3      48
Name: count, dtype: int64


In [37]:
# ============================================================
# 9. AnyAccepted 파생변수 (AcceptedCmp1~5 중 하나라도 1이면 1)
# ============================================================
cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
df['AnyAccepted'] = (df[cmp_cols].sum(axis=1) > 0).astype(int)
print(f"[9] AnyAccepted 분포:\n{df['AnyAccepted'].value_counts()}")

[9] AnyAccepted 분포:
AnyAccepted
0    1613
1     423
Name: count, dtype: int64


In [38]:
# ============================================================
# 최종 확인 및 저장
# ============================================================
print(f"\n=== 최종 데이터 shape: {df.shape} ===")
print(df.dtypes)
 
df.to_csv('marketing_campaign_clean.csv', index=False)
print("\n저장 완료: marketing_campaign_clean.csv")


=== 최종 데이터 shape: (2036, 32) ===
ID                              int64
Year_Birth                      int64
Education                         str
Marital_Status                    str
Income                        float64
Kidhome                         int64
Teenhome                        int64
Dt_Customer            datetime64[us]
Recency                         int64
MntWines                        int64
MntFruits                       int64
MntMeatProducts                 int64
MntFishProducts                 int64
MntSweetProducts                int64
MntGoldProds                    int64
NumDealsPurchases               int64
NumWebPurchases                 int64
NumCatalogPurchases             int64
NumStorePurchases               int64
NumWebVisitsMonth               int64
AcceptedCmp3                    int64
AcceptedCmp4                    int64
AcceptedCmp5                    int64
AcceptedCmp1                    int64
AcceptedCmp2                    int64
Complain        

In [39]:
pd.set_option('display.max_rows', None)
print(df.dtypes)

ID                              int64
Year_Birth                      int64
Education                         str
Marital_Status                    str
Income                        float64
Kidhome                         int64
Teenhome                        int64
Dt_Customer            datetime64[us]
Recency                         int64
MntWines                        int64
MntFruits                       int64
MntMeatProducts                 int64
MntFishProducts                 int64
MntSweetProducts                int64
MntGoldProds                    int64
NumDealsPurchases               int64
NumWebPurchases                 int64
NumCatalogPurchases             int64
NumStorePurchases               int64
NumWebVisitsMonth               int64
AcceptedCmp3                    int64
AcceptedCmp4                    int64
AcceptedCmp5                    int64
AcceptedCmp1                    int64
AcceptedCmp2                    int64
Complain                        int64
Response    